# 03 — Multi-resolution datasets and forecasting sample indices

This notebook constructs the 4, 12, 20, 30, and 60-minute datasets used in the forecasting benchmark. It also creates reusable sample-index files for the 6, 12, 24, and 48-hour historical windows and the 60, 120, 240, and 480-minute forecast horizons.

The workflow uses chronological training, validation, and test partitions. A resolution bin is valid for target evaluation when at least 75% of its underlying observations are original observations. Historical windows may contain no more than 5% interpolated target values and may not cross split boundaries.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from the repository root "
        "or from its notebooks directory."
    )

PROJECT_ROOT = find_project_root()
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "greenhouse_sensor_data_4min.csv"
RESOLUTION_DIR = PROJECT_ROOT / "data" / "processed" / "resolutions"
INDEX_DIR = PROJECT_ROOT / "data" / "processed" / "sample_indices"
RESULTS_DIR = PROJECT_ROOT / "results" / "multiresolution"
FIGURES_DIR = PROJECT_ROOT / "figures" / "multiresolution"
METADATA_DIR = PROJECT_ROOT / "metadata"

for directory in (RESOLUTION_DIR, INDEX_DIR, RESULTS_DIR, FIGURES_DIR, METADATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RESOLUTION_MINUTES = [4, 12, 20, 30, 60]
HISTORY_HOURS = [6, 12, 24, 48]
FORECAST_HORIZONS_MINUTES = [60, 120, 240, 480]
MINIMUM_BIN_COVERAGE = 0.75
MINIMUM_TARGET_OBSERVED_FRACTION = 0.75
MAXIMUM_INTERPOLATED_FRACTION_IN_HISTORY = 0.05

SPLIT_BOUNDARIES = {
    "train": (pd.Timestamp("2026-03-22"), pd.Timestamp("2026-07-05")),
    "validation": (pd.Timestamp("2026-07-05"), pd.Timestamp("2026-07-27")),
    "test": (pd.Timestamp("2026-07-27"), pd.Timestamp("2026-08-20")),
}

FEATURE_SETS = {
    "SHT_BASE": ["temperature", "relative_humidity"],
    "SHT_TIME": [
        "temperature", "relative_humidity", "hour_sin", "hour_cos",
        "day_of_year_sin", "day_of_year_cos",
    ],
    "SHT_AUX_TIME": [
        "temperature", "relative_humidity", "temperature_bme280", "pressure",
        "hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos",
    ],
    "SHT_MULTISENSOR_TIME": [
        "temperature", "relative_humidity", "temperature_bme280",
        "relative_humidity_bme280", "pressure", "rh_bme280_saturated_fraction",
        "hour_sin", "hour_cos", "day_of_year_sin", "day_of_year_cos",
    ],
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()}")
print(f"Input file: {INPUT_FILE.relative_to(PROJECT_ROOT)}")


## 1. Load the quality-controlled 4-minute series and derive microclimate variables


In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Required input file not found: {INPUT_FILE}")

base = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
required_columns = [
    "timestamp", "temp_sht31_clean", "rh_sht31_clean",
    "temp_bme280_clean", "rh_bme280_clean", "pressure_bme280_clean",
    "target_pair_observed_flag", "target_pair_interpolated_flag",
    "target_pair_available_flag", "rh_bme280_saturated_flag",
]
missing_columns = sorted(set(required_columns).difference(base.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

base["timestamp"] = pd.to_datetime(base["timestamp"], errors="coerce")
if base["timestamp"].isna().any():
    raise ValueError("The input contains invalid timestamps.")
base = base.sort_values("timestamp").set_index("timestamp")

intervals = base.index.to_series().diff().dt.total_seconds().dropna()
if not intervals.eq(240).all():
    raise ValueError("The input must be a strictly regular 4-minute time series.")

base["temperature"] = pd.to_numeric(base["temp_sht31_clean"], errors="coerce")
base["relative_humidity"] = pd.to_numeric(base["rh_sht31_clean"], errors="coerce")
base["temperature_bme280"] = pd.to_numeric(base["temp_bme280_clean"], errors="coerce")
base["relative_humidity_bme280"] = pd.to_numeric(base["rh_bme280_clean"], errors="coerce")
base["pressure"] = pd.to_numeric(base["pressure_bme280_clean"], errors="coerce")

temperature = base["temperature"]
relative_humidity = base["relative_humidity"]
saturation_pressure = 0.6108 * np.exp(17.27 * temperature / (temperature + 237.3))
actual_pressure = saturation_pressure * relative_humidity / 100
gamma = np.log(relative_humidity / 100) + 17.27 * temperature / (temperature + 237.3)
base["dew_point_c"] = 237.3 * gamma / (17.27 - gamma)
base["vpd_kpa"] = saturation_pressure - actual_pressure
base["absolute_humidity_g_m3"] = 2167 * actual_pressure / (temperature + 273.15)

flag_columns = [
    "target_pair_observed_flag", "target_pair_interpolated_flag",
    "target_pair_available_flag", "rh_bme280_saturated_flag",
]
base[flag_columns] = base[flag_columns].apply(pd.to_numeric, errors="coerce").fillna(0)

base_verification = pd.DataFrame(
    {
        "indicator": [
            "Rows", "Start", "End", "Minimum interval [s]",
            "Maximum interval [s]", "Duplicate timestamps",
        ],
        "value": [
            len(base), base.index.min(), base.index.max(), intervals.min(), intervals.max(),
            int(base.index.duplicated().sum()),
        ],
    }
)
base_verification.to_csv(RESULTS_DIR / "01_base_frequency_verification.csv", index=False)
display(base_verification)


## 2. Construct the five temporal resolutions

Continuous variables are averaged. Coverage and quality fractions are calculated from the underlying 4-minute observations. Values remain available for model inputs when at least one underlying value exists, while the stricter 75% observed-data rule determines whether a bin may be used as a forecast target.


In [ ]:
CONTINUOUS_VARIABLES = [
    "temperature", "relative_humidity", "temperature_bme280",
    "relative_humidity_bme280", "pressure", "dew_point_c",
    "vpd_kpa", "absolute_humidity_g_m3",
]

def assign_split(index):
    conditions = [
        (index >= SPLIT_BOUNDARIES["train"][0]) & (index < SPLIT_BOUNDARIES["train"][1]),
        (index >= SPLIT_BOUNDARIES["validation"][0]) & (index < SPLIT_BOUNDARIES["validation"][1]),
        (index >= SPLIT_BOUNDARIES["test"][0]) & (index < SPLIT_BOUNDARIES["test"][1]),
    ]
    return np.select(conditions, ["train", "validation", "test"], default="outside")

def add_calendar_features(dataset):
    index = dataset.index
    decimal_hour = index.hour + index.minute / 60
    dataset["year"] = index.year.astype("int32")
    dataset["month_number"] = index.month.astype("int32")
    dataset["day_of_year"] = index.dayofyear.astype("int32")
    dataset["hour"] = index.hour.astype("int32")
    dataset["minute"] = index.minute.astype("int32")
    dataset["hour_sin"] = np.sin(2 * np.pi * decimal_hour / 24)
    dataset["hour_cos"] = np.cos(2 * np.pi * decimal_hour / 24)
    dataset["day_of_year_sin"] = np.sin(2 * np.pi * dataset["day_of_year"] / 365.25)
    dataset["day_of_year_cos"] = np.cos(2 * np.pi * dataset["day_of_year"] / 365.25)
    dataset["split"] = assign_split(index)

def construct_resolution(resolution_minutes):
    frequency = f"{resolution_minutes}min"
    grouped = base.resample(frequency, origin="start_day")
    dataset = pd.DataFrame(index=grouped.size().index)
    dataset.index.name = "timestamp"
    dataset["expected_base_count"] = grouped.size().astype(int)
    for variable in CONTINUOUS_VARIABLES:
        dataset[variable] = grouped[variable].mean()
        dataset[f"{variable}_coverage"] = grouped[variable].count() / dataset["expected_base_count"]
    dataset["target_pair_observed_fraction"] = grouped["target_pair_observed_flag"].mean()
    dataset["target_pair_interpolated_fraction"] = grouped["target_pair_interpolated_flag"].mean()
    dataset["target_pair_available_fraction"] = grouped["target_pair_available_flag"].mean()
    dataset["rh_bme280_saturated_fraction"] = grouped["rh_bme280_saturated_flag"].mean()
    add_calendar_features(dataset)
    dataset["target_valid_for_evaluation_flag"] = (
        dataset["target_pair_observed_fraction"] >= MINIMUM_TARGET_OBSERVED_FRACTION
    )
    dataset["resolution_minutes"] = resolution_minutes
    return dataset

resolution_datasets = {}
resolution_catalog_rows = []
coverage_rows = []
for minutes in RESOLUTION_MINUTES:
    name = f"{minutes}min"
    dataset = construct_resolution(minutes)
    resolution_datasets[name] = dataset
    output_path = RESOLUTION_DIR / f"greenhouse_{name}.csv"
    dataset.reset_index().to_csv(output_path, index=False, encoding="utf-8-sig")
    resolution_catalog_rows.append(
        {
            "resolution": name, "resolution_minutes": minutes, "n_bins": len(dataset),
            "start": dataset.index.min(), "end": dataset.index.max(),
            "target_valid_bins": int(dataset["target_valid_for_evaluation_flag"].sum()),
            "target_valid_pct": float(dataset["target_valid_for_evaluation_flag"].mean() * 100),
            "file": str(output_path.relative_to(PROJECT_ROOT)),
        }
    )
    for variable in CONTINUOUS_VARIABLES:
        coverage_rows.append(
            {
                "resolution": name, "variable": variable, "n_bins": len(dataset),
                "n_valid": int(dataset[variable].notna().sum()),
                "coverage_pct": float(dataset[variable].notna().mean() * 100),
                "mean_base_coverage": float(dataset[f"{variable}_coverage"].mean()),
            }
        )

resolution_catalog = pd.DataFrame(resolution_catalog_rows)
coverage_by_resolution = pd.DataFrame(coverage_rows)
resolution_catalog.to_csv(RESULTS_DIR / "02_resolution_catalog.csv", index=False)
coverage_by_resolution.to_csv(RESULTS_DIR / "03_coverage_by_resolution.csv", index=False)
display(resolution_catalog)


## 3. Convert physical history windows and forecast horizons into steps


In [ ]:
step_rows = []
for minutes in RESOLUTION_MINUTES:
    for history_hours in HISTORY_HOURS:
        row = {
            "resolution": f"{minutes}min", "resolution_minutes": minutes,
            "history_hours": history_hours, "history_steps": history_hours * 60 // minutes,
        }
        for horizon in FORECAST_HORIZONS_MINUTES:
            row[f"horizon_{horizon}_steps"] = horizon // minutes
        step_rows.append(row)
history_horizon_steps = pd.DataFrame(step_rows)
history_horizon_steps.to_csv(RESULTS_DIR / "04_history_and_horizon_steps.csv", index=False)

split_rows = [
    {
        "split": split, "start_inclusive": start.date(), "end_exclusive": end.date(),
        "n_calendar_days": (end - start).days,
    }
    for split, (start, end) in SPLIT_BOUNDARIES.items()
]
split_table = pd.DataFrame(split_rows)
split_table.to_csv(RESULTS_DIR / "05_chronological_split_boundaries.csv", index=False)
display(history_horizon_steps)
display(split_table)


## 4. Build reusable sample-index files

Each index row identifies one forecast origin, its historical window, and the four future target positions. All features must be complete at the requested minimum bin coverage throughout the history. All four targets must be valid original-data bins and remain in the same chronological split.


In [ ]:
INDEX_COLUMNS = [
    "origin_index", "history_start_index", "origin_timestamp", "history_start_timestamp",
    "split", "resolution", "resolution_minutes", "feature_set",
    "history_hours", "history_steps", "n_features", "history_interpolated_fraction",
]
for horizon in FORECAST_HORIZONS_MINUTES:
    INDEX_COLUMNS.extend([f"target_index_h{horizon}", f"target_timestamp_h{horizon}"])

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def build_sample_indices(dataset, resolution, minutes, feature_set, features, history_hours):
    history_steps = history_hours * 60 // minutes
    horizon_steps = {horizon: horizon // minutes for horizon in FORECAST_HORIZONS_MINUTES}
    max_horizon_steps = max(horizon_steps.values())

    row_complete = dataset[features].notna().all(axis=1)
    for feature in features:
        coverage_column = f"{feature}_coverage"
        if coverage_column in dataset.columns:
            row_complete &= dataset[coverage_column].ge(MINIMUM_BIN_COVERAGE)

    history_complete = row_complete.rolling(history_steps, min_periods=history_steps).sum().eq(history_steps)
    history_interpolated = (
        dataset["target_pair_interpolated_fraction"]
        .rolling(history_steps, min_periods=history_steps)
        .mean()
    )

    counters = {
        "total_rows": len(dataset), "enough_context": 0,
        "history_complete_after_context": 0, "interpolation_ok_after_history": 0,
        "same_split_history": 0, "targets_valid": 0,
        "same_split_targets": 0, "valid_samples": 0,
    }
    rows = []
    for origin_index in range(history_steps - 1, len(dataset) - max_horizon_steps):
        counters["enough_context"] += 1
        if not bool(history_complete.iloc[origin_index]):
            continue
        counters["history_complete_after_context"] += 1
        interpolated_fraction = float(history_interpolated.iloc[origin_index])
        if interpolated_fraction > MAXIMUM_INTERPOLATED_FRACTION_IN_HISTORY:
            continue
        counters["interpolation_ok_after_history"] += 1

        history_start_index = origin_index - history_steps + 1
        origin_split = dataset["split"].iloc[origin_index]
        if dataset["split"].iloc[history_start_index] != origin_split:
            continue
        counters["same_split_history"] += 1

        target_indices = {horizon: origin_index + steps for horizon, steps in horizon_steps.items()}
        if not all(bool(dataset["target_valid_for_evaluation_flag"].iloc[index]) for index in target_indices.values()):
            continue
        counters["targets_valid"] += 1
        if not all(dataset["split"].iloc[index] == origin_split for index in target_indices.values()):
            continue
        counters["same_split_targets"] += 1

        row = {
            "origin_index": origin_index, "history_start_index": history_start_index,
            "origin_timestamp": dataset.index[origin_index],
            "history_start_timestamp": dataset.index[history_start_index],
            "split": origin_split, "resolution": resolution,
            "resolution_minutes": minutes, "feature_set": feature_set,
            "history_hours": history_hours, "history_steps": history_steps,
            "n_features": len(features),
            "history_interpolated_fraction": interpolated_fraction,
        }
        for horizon, target_index in target_indices.items():
            row[f"target_index_h{horizon}"] = target_index
            row[f"target_timestamp_h{horizon}"] = dataset.index[target_index]
        rows.append(row)

    counters["valid_samples"] = len(rows)
    return pd.DataFrame(rows, columns=INDEX_COLUMNS), counters

index_catalog_rows = []
diagnostic_rows = []
sample_count_rows = []
for resolution, dataset in resolution_datasets.items():
    minutes = int(resolution.removesuffix("min"))
    for feature_set, features in FEATURE_SETS.items():
        feature_directory = INDEX_DIR / resolution / feature_set
        feature_directory.mkdir(parents=True, exist_ok=True)
        for history_hours in HISTORY_HOURS:
            indices, counters = build_sample_indices(
                dataset, resolution, minutes, feature_set, features, history_hours
            )
            output_path = feature_directory / f"indices_{resolution}_{feature_set}_hist_{history_hours:02d}h.csv"
            indices.to_csv(output_path, index=False, encoding="utf-8-sig")
            index_catalog_rows.append(
                {
                    "resolution": resolution, "feature_set": feature_set,
                    "history_hours": history_hours,
                    "history_steps": history_hours * 60 // minutes,
                    "n_features": len(features), "features": " | ".join(features),
                    "n_samples": len(indices), "file": str(output_path.relative_to(PROJECT_ROOT)),
                    "sha256": sha256_file(output_path),
                }
            )
            diagnostic_rows.append(
                {
                    "resolution": resolution, "feature_set": feature_set,
                    "history_hours": history_hours,
                    "history_steps": history_hours * 60 // minutes, **counters,
                }
            )
            split_counts = indices["split"].value_counts() if not indices.empty else pd.Series(dtype=int)
            for split in ["train", "validation", "test"]:
                sample_count_rows.append(
                    {
                        "resolution": resolution, "feature_set": feature_set,
                        "history_hours": history_hours, "split": split,
                        "n_samples": int(split_counts.get(split, 0)),
                    }
                )

index_catalog = pd.DataFrame(index_catalog_rows)
exclusion_diagnostics = pd.DataFrame(diagnostic_rows)
sample_counts = pd.DataFrame(sample_count_rows)
index_catalog.to_csv(RESULTS_DIR / "06_sample_index_catalog.csv", index=False)
exclusion_diagnostics.to_csv(RESULTS_DIR / "07_sample_exclusion_diagnostics.csv", index=False)
sample_counts.to_csv(RESULTS_DIR / "08_sample_counts_by_configuration.csv", index=False)
display(index_catalog.head(12))
print(f"Index files: {len(index_catalog)}")
print(f"Total samples across configurations: {index_catalog['n_samples'].sum()}")


## 5. Validation, metadata, and diagnostic figure


In [ ]:
validation_rows = []
for resolution, dataset in resolution_datasets.items():
    minutes = int(resolution.removesuffix("min"))
    interval_seconds = dataset.index.to_series().diff().dt.total_seconds().dropna()
    validation_rows.append(
        {
            "resolution": resolution, "n_rows": len(dataset),
            "expected_interval_seconds": minutes * 60,
            "all_intervals_correct": bool(interval_seconds.eq(minutes * 60).all()),
            "duplicate_timestamps": int(dataset.index.duplicated().sum()),
            "target_valid_bins": int(dataset["target_valid_for_evaluation_flag"].sum()),
        }
    )
resolution_validation = pd.DataFrame(validation_rows)
resolution_validation.to_csv(RESULTS_DIR / "09_resolution_validation.csv", index=False)

sample_summary = (
    sample_counts.groupby(["resolution", "split"], as_index=False)["n_samples"].sum()
)
sample_summary.to_csv(RESULTS_DIR / "10_sample_summary_by_resolution.csv", index=False)

configuration = {
    "stage": 3, "stage_name": "Multi-resolution datasets and forecasting sample indices",
    "input_file": str(INPUT_FILE.relative_to(PROJECT_ROOT)),
    "temporal_resolutions_minutes": RESOLUTION_MINUTES,
    "history_windows_hours": HISTORY_HOURS,
    "forecast_horizons_minutes": FORECAST_HORIZONS_MINUTES,
    "continuous_aggregation": "mean", "minimum_bin_coverage": MINIMUM_BIN_COVERAGE,
    "minimum_target_observed_fraction": MINIMUM_TARGET_OBSERVED_FRACTION,
    "maximum_interpolated_fraction_in_history": MAXIMUM_INTERPOLATED_FRACTION_IN_HISTORY,
    "split_boundaries": split_rows, "feature_sets": FEATURE_SETS,
    "storage_strategy": "Base resolution CSV files plus reusable sample-index CSV files",
}
with open(METADATA_DIR / "03_multiresolution_configuration.json", "w", encoding="utf-8") as file:
    json.dump(configuration, file, indent=2, default=str)

plot_data = resolution_catalog.copy()
fig, axis = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_data, x="resolution", y="target_valid_pct", color="steelblue", ax=axis)
axis.set_xlabel("Temporal resolution")
axis.set_ylabel("Target bins valid for evaluation (%)")
axis.set_ylim(0, 100)
axis.set_title("Observed-data target validity by temporal resolution")
for container in axis.containers:
    axis.bar_label(container, fmt="%.1f")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_target_validity_by_resolution.png", dpi=300, bbox_inches="tight")
plt.show()

display(resolution_validation)
print(f"Resolution datasets written to: {RESOLUTION_DIR.relative_to(PROJECT_ROOT)}")
print(f"Sample indices written to: {INDEX_DIR.relative_to(PROJECT_ROOT)}")
print(f"Results written to: {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
